![IITIS](pictures/logoIITISduze.png)

# Wykorzystanie procesorów graficznych (GPU) w algorytmach heurystycznych

wykorzystamy paczkę cupy

## Podstawy programowania GPU

In [2]:
# funkcje pomocniczne

import os
import pandas as pd
import numpy as np
import cupy as cp
import matplotlib.pyplot as plt


test_pegasus = os.path.join("instancje", "Pegasus", "P4_CBFM-P.txt")  # E = -469.0
full_pegasus = os.path.join("instancje", "Pegasus", "P16_CBFM-P.txt")  # E = -12772.0


def read_instance(path: os.PathLike):
    df = pd.read_csv(path, sep=" ", header=None, comment="#", names=["i", "j", "value"])

    n = max(df[["i", "j"]].max())
    h = np.zeros(n)
    J = np.zeros((n, n))
    
    for row in df.itertuples():
        if row.i == row.j:
            h[row.i - 1] = row.value
        elif row.i > row.j:
            J[row.j - 1, row.i - 1] = row.value  # by zachować górnotrójkątność
        else:
            J[row.i - 1, row.j - 1] = row.value
            
    return J, h


def dwave_conv_to_minus_half_convention(J: np.ndarray, h: np.ndarray):
    n = len(h)
    herminian_matrix = np.zeros((n, n))

    # de facto wyciągamy -1/2 przed macierz i zamieniamy ją na hermitowską
    for i in range(n):
        for j in range(i + 1, n):
            J_ij = J[i, j]
            herminian_matrix[i, j] = -J_ij
            herminian_matrix[j, i] = -J_ij

    x = np.random.choice([-1, 1], size=n)
    assert np.array_equal(-2 * x @ J @ x.T, x @ herminian_matrix @ x.T)
    assert np.array_equal(herminian_matrix.T, herminian_matrix)  # wszystkie macierze są rzeczywiste

    new_external_fields = -1 * h
    return herminian_matrix, new_external_fields


def calculate_energy_gpu(J: cp.ndarray, h: cp.ndarray, state: cp.ndarray):
    # Zakładamy, że J jest hermitowska z czynnikiem 1/2
    n, _ = J.shape
    A = cp.multiply(-1/2, J)
    B = cp.matmul(A, state) - h.reshape(n, 1)
    C = cp.multiply(state, B)
    return cp.sum(C, axis=0)


## Wyżarzanie równoległe

In [3]:
# Implementacja algorytmu wyżarzania równoległego używając własnego kernelu w CUDA

import cupy as cp
from typing import Optional
from tqdm import tqdm



def parrarel_annealing_gpu(J, h, step_size: float, lambda_t_max: float, num_steps: int, num_trajectories: int,
                       schedule: Optional[list] = None, dtype = cp.float32):
    
    n = len(h)
    x = cp.zeros((n, num_trajectories), dtype=dtype)  # stan podstawowy dla H_innit = sum(x**2)
    momentum = cp.zeros((n, num_trajectories), dtype=dtype)
    state = cp.random.choice([dtype(-1.), dtype(1.)], size=(n, num_trajectories)) # losowy stan początkowy
    step_size = dtype(step_size)

    if schedule is None:
        schedule = [dtype(lambda_t_max * (1 - i/(num_steps -1))) for i in range(num_steps)]  # dlaczego nie linspace? chodzi o typy

    kernel = cp.RawModule(path="cuda_kernels/pa_kernel.ptx")
    parrarel_annealing_step = kernel.get_function("parrarel_annealing_step")

    threadsperblock = 256  # Ilość wątków w bloku,
    blockspergrid_x = num_trajectories  # każdy blok zajmuje się trajektorią
    blockspergrid_y = (n + threadsperblock - 1) // threadsperblock  # wystarczająca ilość bloków by pomieścić całą kolumnę 
    blockspergrid = (blockspergrid_x, blockspergrid_y)

    x_new = cp.empty_like(x)
    momentum_new = cp.empty_like(momentum)
    state_new = cp.empty_like(state)

    for k in tqdm(range(num_steps), desc="wyżarzanie równoległe GPU"):
   
        lambda_t = schedule[k]
        A = cp.matmul(J, state)
        parrarel_annealing_step(blockspergrid, (threadsperblock,), (A, h, x, momentum, 
                                                               lambda_t, step_size, 
                                                               momentum_new, x_new, state_new))
        momentum = momentum_new
        x = x_new
        state = state_new

    return state, calculate_energy_gpu(J, h, state)


In [4]:
import numpy as np
J, h = read_instance(test_pegasus)
J, h = dwave_conv_to_minus_half_convention(J, h)

J = cp.asarray(J, dtype=cp.float32)
h = cp.asarray(h, dtype=cp.float32)

states, energy = parrarel_annealing_gpu(J, h, step_size=0.01, lambda_t_max=10, num_steps=1000, num_trajectories=64)
print(min(energy))

wyżarzanie równoległe GPU: 100%|██████████| 1000/1000 [00:00<00:00, 1684.39it/s]


-469.0


In [5]:
# E = -12772
# best found -12736 steps 10^4 trajectories 2^10 time approx 43s
# najperw trzeba raz przepuscić by kernel wszedł do pamięci podręcznej (można użyć mało kroków)

J, h = read_instance(full_pegasus)
J, h = dwave_conv_to_minus_half_convention(J, h)

J = cp.asarray(J, dtype=cp.float32)
h = cp.asarray(h, dtype=cp.float32)


states, energy = parrarel_annealing_gpu(J, h, step_size=0.01, lambda_t_max=10, num_steps=10000, num_trajectories=2**10)
print(min(energy))

wyżarzanie równoległe GPU: 100%|██████████| 10000/10000 [00:10<00:00, 933.84it/s]


-26.0


## Bruteforce

In [6]:
# brute force CUDA

import cupy as cp
from tqdm import tqdm

def select_lowest(energies, states, num_states):
    
    indices = cp.argpartition(energies, num_states)[:num_states]

    low_energies = energies[indices]
    low_states = states[indices]
    return low_energies, low_states

def sort_by_key(energies, states, num_states):
    order = cp.argsort(energies)[:num_states]

    low_energies = energies[order]
    low_states = states[order]
    return low_energies, low_states


def brute_force_gpu(Q,  num_states: int, sweep_size_exponent: int = 10):
    N, _ = Q.shape
    sweep_size = 2**sweep_size_exponent
    num_chunks = 2**(N-sweep_size_exponent)

    brute_force_kernel = cp.RawModule(path="cuda_kernels/brute_force_kernel.ptx")
    compute_energies = brute_force_kernel.get_function("compute_energies")

    threadsperblock = 256
    blockspergrid = sweep_size//threadsperblock
    
    final_energies = cp.array([])
    final_states = cp.array([])
    
    for i in tqdm(range(num_chunks), desc="wyczerpujące przeszukiwanie"):

        energies = cp.empty(sweep_size, dtype=cp.float32)
        states = cp.empty(sweep_size, dtype=cp.int64)
        compute_energies((blockspergrid,), (threadsperblock,), 
                         (Q, cp.int32(sweep_size_exponent), energies, states, cp.int32(i)))

        low_energies, low_states = select_lowest(energies, states, num_states)
        
        final_energies = cp.concatenate((final_energies, low_energies))
        final_states = cp.concatenate((final_states, low_states))
        if i != 0:

            final_energies, final_states = select_lowest(final_energies, final_states, num_states)

    return sort_by_key(final_energies, final_states, num_states)

    

In [7]:
# test bruteforce
import cupy as cp
import numpy as np
from itertools import product
from tqdm import tqdm

N = 20
k = 10

brute_force_kernel = cp.RawModule(path="cuda_kernels/brute_force_kernel.ptx")
compute_energies = brute_force_kernel.get_function("compute_energies")


Q = cp.random.uniform(-1, 1, (N, N), dtype=cp.float32)
Q = cp.triu(Q)



energies, states = brute_force_gpu(Q, 10)
print(energies)
print(states)

en = []
st = []
Q2 = cp.asnumpy(Q)
for state in tqdm(product([0, 1], repeat=N), total=2**N):
    s = np.array(state)
    e = s @ Q2 @ s.T
    en.append(e.item())
en.sort()
print(en[:10])

wyczerpujące przeszukiwanie: 100%|██████████| 1024/1024 [00:06<00:00, 161.80it/s]


[-14.9062376  -14.78012276 -14.62869835 -14.37365532 -14.32774925
 -14.25380611 -14.22648239 -14.20959282 -14.16736698 -14.08194351]
[ 822731. 1019275.  953803.  953739.  822735.  888203.  888267.  986507.
  823247. 1019339.]


100%|██████████| 1048576/1048576 [00:06<00:00, 158114.86it/s]

[-14.906237125396729, -14.78012204170227, -14.62869793176651, -14.373656034469604, -14.327750384807587, -14.253807246685028, -14.226483941078186, -14.20959198474884, -14.167365074157715, -14.081944167613983]


# Symulowana bifurkacja 

In [8]:
# Simulated bifurcation GPU naive
import cupy as cp
from math import sqrt
from typing import Optional
from tqdm import tqdm

def wall_gpu(x:cp.ndarray, y:cp.ndarray):
    n, m = x.shape
    for i in range(n):
        for j in range(m):
            if abs(x[i, j]) > 1:
                x[i, j] = cp.sign(x[i, j])
                y[i, j] = 0
    return x, y


def balistic_simulated_bifurcation_gpu_naive(J, h, num_steps, time_step, num_trajectories: int, 
                                   a_0: Optional[float] = None, c_0_scaling: Optional[float] = None):
    if a_0 is None:
        a_0 = 1

    N, _ = J.shape
    mean_J = cp.sqrt(cp.sum(cp.square(J)) / (N * (N - 1)) )
    c_0 = 0.5 / (mean_J * sqrt(N))

    if c_0_scaling is not None:
        c_0 *= c_0_scaling


    a = cp.linspace(0, a_0, num=num_steps)

    x = cp.zeros((N, num_trajectories))
    y = cp.random.uniform(-0.1, 0.1, (N, num_trajectories))

    for t in tqdm(range(num_steps), desc="symulowana bifurkacja"):
        y += (-1 * (a_0 - a[t]) * x + c_0 * cp.matmul(J,x)) * time_step  # x(t)
        x += a_0 * y * time_step # x(t + 1)

        x, y = wall_gpu(x, y)

    x = np.sign(x)
    return x, calculate_energy_gpu(J, h, x)

In [9]:
# Implementacja SBM używając własnego kernelu
import cupy as cp
from math import sqrt
from typing import Optional
from tqdm import tqdm

def balistic_simulated_bifurcation_gpu(J, h, num_steps, time_step, num_trajectories: int, 
                                   a_0: Optional[float] = None, c_0_scaling: Optional[float] = None):
    if a_0 is None:
        a_0 = 1
    
    N, _ = J.shape
    mean_J = np.sqrt(np.sum(np.square(J)) / (N * (N - 1)) )
    c_0 = 0.5 / (mean_J * sqrt(N))

    if c_0_scaling is not None:
        c_0 *= c_0_scaling

    dtype = cp.float32

    a = [dtype(i * a_0 / (num_steps - 1)) for i in range(num_steps)]
    
    a_0 = dtype(a_0)
    c_0 = dtype(c_0.item())
    time_step = dtype(time_step)

    x = cp.zeros((N, num_trajectories), dtype=dtype)
    y = cp.random.uniform(-0.1, 0.1, (N, num_trajectories), dtype=dtype)

    x_new = cp.empty_like(x)
    y_new = cp.empty_like(y)

    threadsperblock = 256  # Ilość wątków w bloku,
    blockspergrid_x = num_trajectories  # każdy blok zajmuje się trajektorią
    blockspergrid_y = (N + threadsperblock - 1) // threadsperblock  # wystarczająca ilość bloków by pomieścić całą kolumnę 
    blockspergrid = (blockspergrid_x, blockspergrid_y)
    
    kernel = cp.RawModule(path="cuda_kernels/sbm_kernel.ptx")
    update_y = kernel.get_function("update_y")
    update_x_and_wall = kernel.get_function("update_x_and_wall")

    for t in tqdm(range(num_steps), desc="symulowana bifurkacja"):
        A = cp.matmul(J, x)
        update_y(blockspergrid, (threadsperblock,),(x, y, A, a_0, a[t], c_0, time_step, y_new))
        y = y_new
        update_x_and_wall(blockspergrid, (threadsperblock,),(x, y, a_0, time_step, x_new, y_new))
        x = x_new
        y = y_new

    x = cp.sign(x)
    return x, calculate_energy_gpu(J, h, x)

In [10]:
J, h = read_instance(test_pegasus)
J, h = dwave_conv_to_minus_half_convention(J, h)

J = cp.asarray(J, dtype=cp.float32)
h = cp.asarray(h, dtype=cp.float32)

states, energy = balistic_simulated_bifurcation_gpu(J, h, time_step=0.5, num_steps=200, num_trajectories=1000, c_0_scaling=0.7)
print(min(energy))

symulowana bifurkacja: 100%|██████████| 200/200 [00:00<00:00, 9012.06it/s]

-465.0


In [11]:
# Implementacja Discrete Simulated Bifurcation

def discrete_simulated_bifurcation_gpu(J, h, num_steps, time_step, num_trajectories: int, 
                                   a_0: Optional[float] = None, c_0_scaling: Optional[float] = None):
    if a_0 is None:
        a_0 = 1
    
    N, _ = J.shape
    mean_J = np.sqrt(np.sum(np.square(J)) / (N * (N - 1)) )
    c_0 = 0.5 / (mean_J * sqrt(N))

    if c_0_scaling is not None:
        c_0 *= c_0_scaling

    dtype = cp.float32

    a = [dtype(i * a_0 / (num_steps - 1)) for i in range(num_steps)]
    
    a_0 = dtype(a_0)
    c_0 = dtype(c_0.item())
    time_step = dtype(time_step)

    x = cp.zeros((N, num_trajectories), dtype=dtype)
    y = cp.random.uniform(-0.1, 0.1, (N, num_trajectories), dtype=dtype)

    x_new = cp.empty_like(x)
    y_new = cp.empty_like(y)

    threadsperblock = 256  # Ilość wątków w bloku,
    blockspergrid_x = num_trajectories  # każdy blok zajmuje się trajektorią
    blockspergrid_y = (N + threadsperblock - 1) // threadsperblock  # wystarczająca ilość bloków by pomieścić całą kolumnę 
    blockspergrid = (blockspergrid_x, blockspergrid_y)
    
    kernel = cp.RawModule(path="cuda_kernels/sbm_kernel.ptx")
    update_y = kernel.get_function("update_y")
    update_x_and_wall = kernel.get_function("update_x_and_wall")

    for t in tqdm(range(num_steps), desc="symulowana bifurkacja"):
        sign_x = cp.sign(x)
        A = cp.matmul(J, sign_x)
        update_y(blockspergrid, (threadsperblock,),(x, y, A, a_0, a[t], c_0, time_step, y_new))
        y = y_new
        update_x_and_wall(blockspergrid, (threadsperblock,),(x, y, a_0, time_step, x_new, y_new))
        x = x_new
        y = y_new

    x = cp.sign(x)
    return x, calculate_energy_gpu(J, h, x)


In [12]:
J, h = read_instance(test_pegasus)
J, h = dwave_conv_to_minus_half_convention(J, h)

J = cp.asarray(J, dtype=cp.float32)
h = cp.asarray(h, dtype=cp.float32)

states, energy = discrete_simulated_bifurcation_gpu(J, h, time_step=0.5, num_steps=2000, num_trajectories=1000, c_0_scaling=0.7)
print(min(energy))

symulowana bifurkacja:   0%|          | 0/2000 [00:00<?, ?it/s]

symulowana bifurkacja: 100%|██████████| 2000/2000 [00:00<00:00, 7151.71it/s]


-465.0


In [13]:
# Test

import numpy as np
import cupy as cp

dtype = cp.float32
threadsperblock = 256
N = 5
num_trajectories = 2


J = cp.random.uniform(-1, 1, (N, N), dtype=dtype)
x = cp.random.uniform(-1, 1, (N, num_trajectories), dtype=dtype)
y = cp.random.uniform(-1, 1, (N, num_trajectories), dtype=dtype)
x_new = cp.empty_like(x)
y_new = cp.empty_like(y)
a_0 = cp.float32(1.0)
a_t = cp.float32(0.1)
time_step = cp.float32(0.25)
c_0 = cp.float32(0.25)


blockspergrid_x = num_trajectories  # każdy blok zajmuje się trajektorią
blockspergrid_y = (N + threadsperblock - 1) // threadsperblock  # wystarczająca ilość bloków by pomieścić całą kolumnę 
blockspergrid = (blockspergrid_x, blockspergrid_y)

kernel = cp.RawModule(path="cuda_kernels/sbm_kernel.ptx")
update_y = kernel.get_function("update_y")
# #update_x_and_wall = kernel.get_function("update_x_and_wall")

A = cp.matmul(J,x)


update_y(blockspergrid, (threadsperblock,),(x, y, A, a_0, a_t, c_0, time_step, y_new))
y = y_new

# update_x_and_wall(blockspergrid, (threadsperblock,),(x, y, a_0, time_step, x_new, y_new))
# x = x_new
# y = y_new
# print(x)
# print(y)

# 